In [1]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from metrics import apk

torch.manual_seed(42)
np.random.seed(42)

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")

# Item vocabulary: index 0 is reserved for "padding" (empty slot)
items = con.execute("SELECT DISTINCT article_id FROM tx ORDER BY article_id").df()["article_id"].tolist()
item2idx = {a: i + 1 for i, a in enumerate(items)}
n_items = len(items) + 1

# Text embeddings in the same order (row 0 = zeros for padding)
all_emb = np.load(DATA / "article_emb.npy")
all_ids = np.load(DATA / "article_emb_ids.npy")
row_of = {a: i for i, a in enumerate(all_ids)}
text_emb = np.zeros((n_items, all_emb.shape[1]), dtype=np.float32)
for a, i in item2idx.items():
    text_emb[i] = all_emb[row_of[a]]

print(f"Items in vocabulary: {n_items - 1:,}  (+1 padding)")
print(f"Text embedding matrix: {text_emb.shape}")

Items in vocabulary: 39,709  (+1 padding)
Text embedding matrix: (39710, 384)


In [2]:
MAX_HIST = 20

def history(first_w, last_w):
    df = con.execute(f"""
        SELECT customer_id,
               LIST(article_id ORDER BY last_buy DESC, article_id) AS items
        FROM (SELECT customer_id, article_id, MAX(t_dat) AS last_buy
              FROM tx WHERE w BETWEEN {first_w} AND {last_w}
              GROUP BY customer_id, article_id)
        GROUP BY customer_id
    """).df()
    return {c: [item2idx[a] for a in i][:MAX_HIST] for c, i in zip(df["customer_id"], df["items"])}

def pad(seq):
    return seq + [0] * (MAX_HIST - len(seq))

H, P = [], []
for t in [2, 3, 4, 5]:
    hist = history(t + 1, t + 8)
    tgt = con.execute(f"SELECT DISTINCT customer_id, article_id FROM tx WHERE w = {t}").df()
    kept = 0
    for c, a in zip(tgt["customer_id"], tgt["article_id"]):
        if c in hist:
            H.append(pad(hist[c])); P.append(item2idx[a]); kept += 1
    print(f"target week {t}: {len(tgt):,} pairs, {kept:,} with history")

H = torch.tensor(H, dtype=torch.long)
P = torch.tensor(P, dtype=torch.long)
print(f"\nTraining examples: {len(P):,} | history matrix {tuple(H.shape)}")

target week 2: 47,760 pairs, 29,253 with history
target week 3: 50,346 pairs, 31,050 with history
target week 4: 45,766 pairs, 29,221 with history
target week 5: 47,483 pairs, 32,310 with history

Training examples: 121,834 | history matrix (121834, 20)


In [3]:
class TwoTower(nn.Module):
    def __init__(self, n_items, text_emb, dim=64):
        super().__init__()
        self.id_emb = nn.Embedding(n_items, dim, padding_idx=0)
        nn.init.normal_(self.id_emb.weight, std=0.01)
        self.register_buffer("text", torch.tensor(text_emb))
        self.text_proj = nn.Linear(text_emb.shape[1], dim)
        self.user_mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))

    def item_vec(self, idx):
        v = self.id_emb(idx) + self.text_proj(self.text[idx])
        return F.normalize(v, dim=-1)

    def user_vec(self, hist):
        mask = (hist > 0).float().unsqueeze(-1)
        v = self.item_vec(hist) * mask
        mean = v.sum(1) / mask.sum(1).clamp(min=1)
        return F.normalize(self.user_mlp(mean), dim=-1)

model = TwoTower(n_items, text_emb)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Quick shape test on 4 examples
u = model.user_vec(H[:4]); i = model.item_vec(P[:4])
print("User vectors:", tuple(u.shape), "| Item vectors:", tuple(i.shape))

Trainable parameters: 2,574,400
User vectors: (4, 64) | Item vectors: (4, 64)


In [4]:
labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}
v_custs = list(actual)

idx2item = {i: a for a, i in item2idx.items()}
vhist = history(2, 9)                                   # weeks 2-9 only
past = {c: [idx2item[i] for i in v] for c, v in vhist.items()}

pool_ids = con.execute("SELECT DISTINCT article_id FROM tx WHERE w BETWEEN 2 AND 3 ORDER BY article_id").df()["article_id"].tolist()
pool_idx = torch.tensor([item2idx[a] for a in pool_ids])
top2 = con.execute("""SELECT article_id FROM tx WHERE w BETWEEN 2 AND 3
    GROUP BY article_id ORDER BY COUNT(DISTINCT customer_id) DESC, article_id LIMIT 12""").df()["article_id"].tolist()

def fill(items, filler, k=12):
    out = list(items[:k])
    for f in filler:
        if len(out) >= k: break
        if f not in out: out.append(f)
    return out

def tt_lists(model, k=50):
    model.eval()
    cs = [c for c in v_custs if c in vhist]
    out = {}
    with torch.no_grad():
        iv = model.item_vec(pool_idx)
        for s in range(0, len(cs), 4096):
            batch = cs[s:s + 4096]
            hb = torch.tensor([pad(vhist[c]) for c in batch])
            top = (model.user_vec(hb) @ iv.T).topk(k, dim=1).indices
            for j, c in enumerate(batch):
                out[c] = [pool_ids[t] for t in top[j].tolist()]
    return out

def mix(c, tl, n=4):
    items = list(past.get(c, [])[:12]); added = 0
    for a in tl.get(c, []):
        if added >= n or len(items) >= 12: break
        if a not in items: items.append(a); added += 1
    return fill(items, top2)

def evaluate(model):
    tl = tt_lists(model)
    ap_pure = np.array([apk(actual[c], fill(tl.get(c, []), top2)) for c in v_custs])
    ap_mix  = np.array([apk(actual[c], mix(c, tl)) for c in v_custs])
    return ap_pure, ap_mix

ap_base = np.array([apk(actual[c], fill(past.get(c, []), top2)) for c in v_custs])
ap_pure, ap_mix = evaluate(model)
print(f"Bought-before baseline: {ap_base.mean():.5f}")
print(f"Untrained two-tower  -> pure {ap_pure.mean():.5f} | mix {ap_mix.mean():.5f}")

Bought-before baseline: 0.02433
Untrained two-tower  -> pure 0.00237 | mix 0.02362


In [5]:
import copy

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
TEMP, BS, EPOCHS = 0.05, 1024, 5
best_mix, best_state, log = -1, None, []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(P))
    total = 0.0
    for s in range(0, len(P), BS):
        b = perm[s:s + BS]
        u = model.user_vec(H[b])
        v = model.item_vec(P[b])
        loss = F.cross_entropy(u @ v.T / TEMP, torch.arange(len(b)))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(b)

    ap_pure, ap_mix = evaluate(model)
    log.append((epoch, total / len(P), ap_pure.mean(), ap_mix.mean()))
    print(f"epoch {epoch} | loss {total/len(P):.3f} | pure {ap_pure.mean():.5f} "
          f"| mix {ap_mix.mean():.5f} | {time.time()-t0:.0f}s")
    if ap_mix.mean() > best_mix:
        best_mix, best_state = ap_mix.mean(), copy.deepcopy(model.state_dict())

print(f"\nBest mix MAP@12: {best_mix:.5f}  (baseline {ap_base.mean():.5f})")

epoch 1 | loss 6.945 | pure 0.00232 | mix 0.02360 | 5s
epoch 2 | loss 6.644 | pure 0.00388 | mix 0.02371 | 4s
epoch 3 | loss 6.003 | pure 0.00477 | mix 0.02367 | 4s
epoch 4 | loss 5.277 | pure 0.00494 | mix 0.02369 | 4s
epoch 5 | loss 4.684 | pure 0.00487 | mix 0.02363 | 4s

Best mix MAP@12: 0.02371  (baseline 0.02433)


In [6]:
def train_eval(temp, lr, dim=64, epochs=8, seed=42):
    torch.manual_seed(seed)
    m = TwoTower(n_items, text_emb, dim=dim)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    best = {"pure": 0, "mix1": 0, "mix2": 0, "mix4": 0}
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(P))
        for s in range(0, len(P), BS):
            b = perm[s:s + BS]
            loss = F.cross_entropy(m.user_vec(H[b]) @ m.item_vec(P[b]).T / temp, torch.arange(len(b)))
            o.zero_grad(); loss.backward(); o.step()
        tl = tt_lists(m)
        scores = {
            "pure": np.mean([apk(actual[c], fill(tl.get(c, []), top2)) for c in v_custs]),
            "mix1": np.mean([apk(actual[c], mix(c, tl, 1)) for c in v_custs]),
            "mix2": np.mean([apk(actual[c], mix(c, tl, 2)) for c in v_custs]),
            "mix4": np.mean([apk(actual[c], mix(c, tl, 4)) for c in v_custs]),
        }
        for k in best: best[k] = max(best[k], scores[k])
    return best

rows = []
for temp, lr in [(0.05, 1e-3), (0.1, 1e-3), (0.05, 3e-3)]:
    t0 = time.time()
    r = train_eval(temp, lr)
    rows.append({"temp": temp, "lr": lr, **{k: round(v, 5) for k, v in r.items()},
                 "secs": round(time.time() - t0)})
    print(rows[-1])

print(f"\nBaseline (bought-before): {ap_base.mean():.5f}")

{'temp': 0.05, 'lr': 0.001, 'pure': np.float64(0.00568), 'mix1': np.float64(0.02411), 'mix2': np.float64(0.02392), 'mix4': np.float64(0.02371), 'secs': 35}
{'temp': 0.1, 'lr': 0.001, 'pure': np.float64(0.00733), 'mix1': np.float64(0.02416), 'mix2': np.float64(0.02402), 'mix4': np.float64(0.02382), 'secs': 36}
{'temp': 0.05, 'lr': 0.003, 'pure': np.float64(0.00462), 'mix1': np.float64(0.02412), 'mix2': np.float64(0.02392), 'mix4': np.float64(0.02369), 'secs': 41}

Baseline (bought-before): 0.02433


## Two-tower model in PyTorch

I built a two-tower model. One tower turns a customer's last 8 weeks of purchases into a vector. The other turns an item into a vector by adding a learned ID vector to a squeezed version of its text embedding from the previous notebook. Training pushes a customer's vector close to the items they buy next week. I trained on four target weeks (2 to 5), always with history from older weeks only, which gave 121,834 examples and 2.57 million parameters.

Before training, I checked that the untrained model scored badly (0.00237), which it did. So nothing was leaking.

The model clearly learned something. The loss dropped from 6.9, which is what random guessing gives, to about 4.7. Its own top 12 picks improved from 0.00232 to 0.00494 after 4 epochs.

But it didn't beat the simple methods. I tried three settings for 8 epochs each. The best one reached 0.00733 on its own, a little above the popularity baseline (0.00686), but inside popularity's noise range and picked as the best epoch on validation, so I wouldn't call it a real win. When I added its picks after a customer's past purchases, the score always went down compared to past purchases alone (0.02416 at best vs 0.02433). Adding fewer of its items hurt less, which tells me its picks are mostly worse than the best-sellers they replace.

My guess at why: 121,834 training examples is small for this kind of model, and in-batch training doesn't directly learn how popular an item is, which turned out to be one of the strongest signals in this data. With a GPU and more customers it might do better. On my laptop with this sample, the simple co-purchase count is the better choice, and I'd rather report that than tune until something looks good.